In [3]:
from langchain_openai import ChatOpenAI

# model = ChatOpenAI(model="gpt-4o-mini")
model = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Crie uma frase sobre: {assunto}")

In [5]:
chain = prompt | model

In [6]:
chain

ChatPromptTemplate(input_variables=['assunto'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['assunto'], template='Crie uma frase sobre: {assunto}'))])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x116983110>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11698b4d0>, model_name='gpt-3.5-turbo-0125', openai_api_key=SecretStr('**********'), openai_proxy='')

In [7]:
chain.invoke({"assunto":"Python"})

AIMessage(content='Python é uma linguagem de programação versátil e poderosa, capaz de tornar a vida dos programadores mais fácil e eficiente.', response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 14, 'total_tokens': 45, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-5d56b8b6-707a-44b2-b9e3-8fdcd94cd8e0-0')

In [8]:
from langchain_core.output_parsers import StrOutputParser

outputParser = StrOutputParser()

chain = prompt | model | outputParser

In [10]:
chain.invoke({"assunto": "Python"})

'Python é uma linguagem de programação versátil e poderosa, capaz de tornar qualquer ideia em realidade.'

In [11]:
input = {"assunto": "Python"}

In [12]:
prompt_format = prompt.invoke(input)
prompt_format

ChatPromptValue(messages=[HumanMessage(content='Crie uma frase sobre: Python')])

In [13]:
resposta = model.invoke(prompt_format)
resposta

AIMessage(content='Python é uma linguagem de programação versátil e poderosa, capaz de tornar a codificação mais simples e eficiente.', response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 14, 'total_tokens': 43, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-5afa5326-f295-4286-9c8a-4a5ee306d3f4-0')

In [14]:
from langchain.chains.llm import LLMChain
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

model = ChatOpenAI(model="gpt-3.5-turbo-0125")

prompt = ChatPromptTemplate.from_template("Crie uma frase sobre o assunto {assunto}")

outputParser = StrOutputParser()

In [15]:
chain = LLMChain(
    llm=model,
    prompt=prompt,
    output_parser=outputParser
)

chain.invoke({"assunto": "Python"})

/Users/luizfelipew/Documents/git/AI/Exploring-LLMs/agents/agentes-ai-apps/.venv/lib/python3.13/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 0.3.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  warn_deprecated(


{'assunto': 'Python',
 'text': 'Python é uma linguagem de programação versátil e poderosa, que torna a codificação mais simples e rápida.'}

In [16]:
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [18]:
caminhos = [
    "files/LLM.pdf"
]

paginas = []
for caminho in caminhos:
    loader = PyPDFLoader(caminho)
    paginas.extend(loader.load())


recur_split = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", ""]
)

documents = recur_split.split_documents(paginas)


Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)
Ignoring wrong pointing object 91 0 (offset 0)
Ignoring wrong pointing object 103 0 (offset 0)
Ignoring wrong pointing object 108 0 (offset 0)
Ignoring wrong pointing object 149 0 (offset 0)


Ignoring wrong pointing object 155 0 (offset 0)
Ignoring wrong pointing object 158 0 (offset 0)
Ignoring wrong pointing object 160 0 (offset 0)
Ignoring wrong pointing object 163 0 (offset 0)
Ignoring wrong pointing object 165 0 (offset 0)


In [20]:
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings()

In [21]:
from langchain_community.vectorstores.faiss import FAISS

vectorestore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings_model
)

retriever = vectorestore.as_retriever(seach_type="mmr")

In [24]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI()

template_str = """Responda as perguntas do usuário com base no contexto fornecido.

Contexto: {context}

Pergunta: {question}

Resposta
"""

template = ChatPromptTemplate.from_template(template_str)
outputParser = StrOutputParser()

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

setup_retrieval = RunnableParallel(
    {
        "pergunta": RunnablePassthrough(),
        "context": retriever
    }
)

chain = setup_retrieval | template | model | outputParser